In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

PDFS_PATH = str(REPO_ROOT / "data" / "mcq" / "pdf")
OUTPUT_FILE = "./questions.json"
MIN_IMAGE_BYTES = 5000

MINERU_BIN = REPO_ROOT / ".venv-mineru" / "bin" / "mineru"
assert MINERU_BIN.exists(), (
    f"MinerU venv not found at {MINERU_BIN}. Create it with:\n"
    f"  uv venv .venv-mineru --python 3.13\n"
    f'  uv pip install --python .venv-mineru "mineru[core]"'
)
MINERU_OUTPUT_ROOT = REPO_ROOT / "output" / "mcq"

In [ ]:
# MinerU layout/OCR pass, same shape as Thread A in dual_pipeline_parsing.ipynb:
# runs via the separate .venv-mineru/ venv (transformers pin conflict with this
# project's env — see that notebook's docstring), one subprocess call per PDF.
#
# Unlike the old per-page Qwen2.5-VL vision pass (whole rendered page image ->
# LLM does OCR + segmentation in one call, ~35s/page on MPS), MinerU's own
# layout+OCR replaces the vision call's OCR duty entirely: it's a purpose-built
# layout/OCR pipeline, not a general vision-language model doing OCR as a side
# effect, so it's dramatically faster per page. The LLM step downstream only
# has to segment/structure already-correct text (see the text-only extractor
# cell below) instead of also reading pixels.
#
# Trade-off: MinerU's content_list carries plain text + bbox only, no bold/
# color/circle style flags (verified against a real _content_list_v2.json —
# text items are just {"type": "text", "content": "..."}). Answers marked
# only by bold/circled formatting with no accompanying "Jawaban: X" / "ANSWER:
# X" text are therefore undetectable from this pipeline and fall into the same
# "answer: null — fill manually" bucket the notebook already reports.

import subprocess

from ingestion.clean_content_list import build_clean_content_list


def run_mineru(pdf_path: Path) -> Path:
    """Runs MinerU on one PDF, returns the path to its _content_list_v2.json."""
    cmd = [
        str(MINERU_BIN),
        "-p", str(pdf_path),
        "-o", str(MINERU_OUTPUT_ROOT),
        "-b", "pipeline",
        "-m", "auto",
    ]
    result = subprocess.run(cmd, cwd=REPO_ROOT, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-2000:])
        print(result.stderr[-2000:])
        raise RuntimeError(f"MinerU failed on {pdf_path.name} (exit {result.returncode})")

    run_dir = MINERU_OUTPUT_ROOT / pdf_path.stem / "auto"
    return run_dir / f"{pdf_path.stem}_content_list_v2.json"


def load_clean_content_list(content_list_v2_path: Path) -> list[dict]:
    clean_path = build_clean_content_list(content_list_v2_path)
    with open(clean_path) as f:
        return json.load(f)

print("MinerU runner defined.")

In [ ]:
# Text-only Qwen3.5-27B page-question extractor: reads a page's already-clean
# MinerU text (not an image) and emits the structured question(s) on that
# page in one pass. Same model/loading pattern as mcq_generation.ipynb's
# TripleExtractor — Qwen3.5 is a multimodal checkpoint (AutoModelForMultimodalLM
# + AutoProcessor) but runs fine text-only, just omit image content from the
# messages. enable_thinking=False since this is structured extraction, not
# open-ended reasoning.
#
# This replaces the old vision call entirely: no page render, no image
# tokens, no OCR-from-pixels — MinerU already did layout+OCR upstream, so
# this call only has to segment/structure text it can trust verbatim.

import gc
import time

import torch
from transformers import AutoModelForMultimodalLM, AutoProcessor

TEXT_MODEL_PATH = "Qwen/Qwen3.5-27B"


def get_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


class PageQuestionExtractor:
    def __init__(self, model_path: str = TEXT_MODEL_PATH):
        self.model_path = model_path
        self.device = get_device()
        self.model = None
        self.processor = None

    def load(self):
        if self.model is not None:
            print("Model already loaded, skipping.")
            return
        print(f"Loading processor for {self.model_path}...")
        self.processor = AutoProcessor.from_pretrained(self.model_path)
        print(f"Loading model weights for {self.model_path}...")
        t0 = time.monotonic()
        self.model = AutoModelForMultimodalLM.from_pretrained(
            self.model_path,
            dtype=torch.bfloat16,
        )
        print(f"Weights loaded in {time.monotonic() - t0:.1f}s, moving to {self.device}...")
        self.model.to(self.device)
        self.model.eval()
        print("Model ready.")

    def unload(self):
        self.model = None
        self.processor = None
        gc.collect()
        if torch.backends.mps.is_available():
            torch.mps.empty_cache()
        elif torch.cuda.is_available():
            torch.cuda.empty_cache()

    def generate(self, prompt: str, max_new_tokens: int = 1536) -> str:
        assert self.model is not None, "call load() first"
        messages = [{"role": "user", "content": prompt}]
        inputs = self.processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            enable_thinking=False,
        ).to(self.device)
        with torch.inference_mode():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=None,
                top_p=None,
                top_k=None,
            )
        new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
        return self.processor.decode(new_tokens, skip_special_tokens=True)


page_extractor = PageQuestionExtractor()
page_extractor.load()
print(f"Loaded {TEXT_MODEL_PATH} on {page_extractor.device}")
if page_extractor.device != "mps" and torch.backends.mps.is_available():
    print("WARNING: MPS is available but model is not using it — check get_device() / device placement.")

In [ ]:
import json
import re
import time
from collections import defaultdict

import fitz
from tqdm.auto import tqdm

from ingestion.unify import build_unified_items, render_unified_markdown

# ── image crop extraction (kept as-is: questions still need has_image/
# image_paths for embedded figures; independent of the text-extraction
# swap above) ───────────────────────────────────────────────────────────
def save_images(doc, out_dir: Path, stem: str) -> list:
    saved, seen = [], set()
    (out_dir / "images").mkdir(parents=True, exist_ok=True)
    for pnum, page in enumerate(doc):
        for img in page.get_images(full=True):
            xref = img[0]
            if xref in seen: continue
            seen.add(xref)
            try:
                bi = doc.extract_image(xref)
                if not bi or bi["size"] < MIN_IMAGE_BYTES: continue
                fname = f"{stem}_p{pnum+1}_x{xref}.{bi['ext']}"
                fpath = out_dir / "images" / fname
                fpath.write_bytes(bi["image"])
                saved.append({"page": pnum, "path": f"images/{fname}"})
            except Exception:
                pass
    return saved


# ── page text grouping ──────────────────────────────────────────────────
# MinerU's clean content_list is a flat, page-tagged item list (title,
# paragraph, list, table, ... each carrying page_idx). Group by page_idx
# and render each page's items as markdown via ingestion.unify — the same
# renderer used by the dual-pipeline document builder — instead of a page
# image, since MinerU already resolved layout/reading order/OCR upstream.
def group_text_by_page(content_list: list[dict], pdf_stem: str) -> dict[int, str]:
    items = build_unified_items(content_list, pdf_stem, captions={})
    by_page: dict[int, list] = defaultdict(list)
    for item in items:
        by_page[item.page_idx].append(item)
    return {page_idx: render_unified_markdown(page_items) for page_idx, page_items in by_page.items()}


# ── LLM-based page parsing ─────────────────────────────────────────────────
# One text-LLM call per page replaces both the old fitz line/bold
# heuristic segmenter AND the vision-LLM-reads-the-page-image approach:
# MinerU has already turned the page into clean, ordered text, so the
# model's only job here is to segment/structure that text into questions
# — no OCR, no layout reconstruction, no image tokens.

PAGE_PARSE_PROMPT = """The following is the MinerU-extracted text (in reading order) of one page from an Indonesian medical multiple-choice exam PDF. Find every question on this page and extract it.

Return ONLY a JSON array, no markdown fences, no commentary. Each element has this exact shape:
{
  "number": <the question's printed number, or null if this is a continuation (see below)>,
  "background": "the clinical vignette/scenario text (patient history, exam findings, case setup), or empty string if this question has no separate scenario",
  "question": "the actual question being asked — the sentence that ends with what to answer (often starts with 'Apakah', 'Bagaimana', 'Manakah', 'Apa')",
  "options": {"A": "...", "B": "...", ...},
  "answer": "A" | null,
  "reference": "citation/source text if present (e.g. 'Referensi: ...' book/journal/author/page), else empty string",
  "has_image": true | false
}

Rules:
- "answer" is the correct option letter ONLY if explicitly marked in the text (e.g. an explicit "ANSWER: X" / "Jawaban: X" line, or an option's text is clearly annotated as correct). Use null if no explicit marker is present in the text — do not guess from medical knowledge, and note that bold/circled/highlighted formatting is NOT preserved in this text so it cannot be used as a signal.
- Options must be transcribed verbatim, just trimmed.
- Split the vignette from the ask: "background" is the case setup (patient demographics, history, exam/lab findings) — usually starts with "Seorang", "Seseorang", "Pasien", "Pada ...". "question" is just the final question sentence itself (e.g. "Apakah diagnosis yang paling mungkin?"), not the case details. Many questions have no real background (a bare factual/conceptual question) — in that case set "background" to "" and put the whole question text in "question". Keep both separate from any reference/citation text even if adjacent in the text.
- "has_image" is true if the text references a photo, diagram, chart, or scan belonging to the question (e.g. a "[FIGURE:...]" marker), not counting decorative page headers/logos.
- If the FIRST question in this text is a continuation of a question whose background/question/options started on the PREVIOUS page (e.g. this page starts mid-option-list with no background/question text, or starts with "Referensi:" / an "ANSWER:" line with no preceding text), set "number" to null for that entry and put whatever text appears on THIS page into its fields — it will be merged with the previous page's last question.
- Ignore page numbers, "Paket X" headers/footers, and running headers unrelated to question content.
- If the page has no questions at all (e.g. a cover page), return an empty array []."""

_THINK_BLOCK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)
_JSON_ARRAY_RE = re.compile(r"\[.*\]", re.DOTALL)


def _repair_truncated_json_array(raw: str) -> str:
    """Best-effort fix for a JSON array cut off mid-generation
    (max_new_tokens truncation): drops the last, possibly-incomplete
    element and closes the array, so json.loads can recover every fully
    formed question that came before it."""
    text = raw[raw.index("["):] if "[" in raw else raw

    if text.count('"') % 2 == 1:
        text = text[: text.rindex('"')]

    depth = 0
    last_complete_end = None
    in_string = False
    escape = False
    for i, ch in enumerate(text):
        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
            continue
        if ch == '"':
            in_string = True
        elif ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                last_complete_end = i

    if last_complete_end is None:
        raise ValueError("no complete element found")
    return text[: last_complete_end + 1] + "]"


def _call_llm_json_array(page_text: str, log_prefix: str) -> list:
    prompt = PAGE_PARSE_PROMPT + "\n\nPAGE TEXT:\n" + page_text

    tqdm.write(f"{log_prefix} calling LLM (generate)...")
    t0 = time.monotonic()
    raw = page_extractor.generate(prompt)
    dt = time.monotonic() - t0
    tqdm.write(f"{log_prefix} LLM generate done in {dt:.1f}s, {len(raw)} chars raw output — parsing JSON...")

    cleaned = _THINK_BLOCK_RE.sub("", raw)
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", cleaned.strip())

    matches = _JSON_ARRAY_RE.findall(cleaned)
    if not matches:
        raise ValueError(f"No JSON array found in LLM output: {raw[:200]!r}")
    candidate = matches[-1]

    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        tqdm.write(f"{log_prefix} JSON truncated, attempting repair...")
        return json.loads(_repair_truncated_json_array(candidate))


def _normalize_item(item: dict) -> dict:
    options = {str(k).upper(): str(v).strip()
               for k, v in (item.get("options") or {}).items()
               if str(k).upper() in "ABCDE"}
    answer = item.get("answer")
    if answer:
        answer = str(answer).strip().upper()
        if answer not in "ABCDE" or answer not in options:
            answer = None
    return {
        "background": str(item.get("background") or "").strip(),
        "question": str(item.get("question") or "").strip(),
        "options": {k: options[k] for k in sorted(options)},
        "answer": answer,
        "reference": str(item.get("reference") or "").strip(),
        "has_image": bool(item.get("has_image")),
    }


def _merge_continuation(prev: dict, cont: dict) -> None:
    """Folds a continuation entry (number == null) into the previous
    question in place: fills in whatever fields the previous page's parse
    left empty, and appends any extra options/reference/answer found."""
    if not prev["background"] and cont["background"]:
        prev["background"] = cont["background"]
    if not prev["question"] and cont["question"]:
        prev["question"] = cont["question"]
    for k, v in cont["options"].items():
        prev["options"].setdefault(k, v)
    prev["options"] = {k: prev["options"][k] for k in sorted(prev["options"])}
    if not prev["answer"] and cont["answer"]:
        prev["answer"] = cont["answer"]
    if cont["reference"]:
        prev["reference"] = (prev["reference"] + " " + cont["reference"]).strip()
    prev["has_image"] = prev["has_image"] or cont["has_image"]


# ── main function ─────────────────────────────────────────────────────────
def extract_pdf(pdf_path: str, out_dir: Path) -> list:
    file_stem = Path(pdf_path).stem
    tqdm.write(f"\n[{file_stem}] opening PDF...")
    try:
        doc = fitz.open(pdf_path)
    except Exception as e:
        tqdm.write(f"  [skip] {Path(pdf_path).name}: {e}")
        return []

    tqdm.write(f"[{file_stem}] {len(doc)} pages — extracting embedded images...")
    images = save_images(doc, out_dir, file_stem)
    n_pages = len(doc)
    doc.close()
    tqdm.write(f"[{file_stem}] {len(images)} embedded image(s) saved")
    images_by_page: dict[int, list] = {}
    for img in images:
        images_by_page.setdefault(img["page"], []).append(img["path"])

    tqdm.write(f"[{file_stem}] running MinerU (layout + OCR)...")
    content_list_v2_path = run_mineru(Path(pdf_path))
    content_list = load_clean_content_list(content_list_v2_path)
    text_by_page = group_text_by_page(content_list, file_stem)

    parsed_questions: list[dict] = []
    for pnum in tqdm(range(n_pages), desc=file_stem, unit="pg", leave=False):
        log_prefix = f"[{file_stem}] page {pnum + 1}/{n_pages}:"
        page_text = text_by_page.get(pnum, "").strip()
        if not page_text:
            tqdm.write(f"{log_prefix} no MinerU text, skipping")
            continue
        try:
            items = _call_llm_json_array(page_text, log_prefix)
        except Exception as e:
            tqdm.write(f"  [warn] LLM parse failed for {file_stem} p{pnum+1}: {e}")
            continue

        tqdm.write(f"{log_prefix} {len(items)} item(s) found")
        page_img_paths = images_by_page.get(pnum, [])
        for raw_item in items:
            norm = _normalize_item(raw_item)
            if raw_item.get("number") is None and parsed_questions:
                tqdm.write(f"{log_prefix} merging continuation into previous question")
                _merge_continuation(parsed_questions[-1], norm)
                if page_img_paths:
                    parsed_questions[-1]["image_paths"].extend(page_img_paths)
            else:
                norm["image_paths"] = list(page_img_paths) if norm["has_image"] else []
                parsed_questions.append(norm)

    questions = []
    for idx, q in enumerate(parsed_questions, 1):
        if not (q["options"] and (q["background"] or q["question"] or q["has_image"])):
            continue
        questions.append({
            "id": f"{file_stem}_Q{idx:03d}",
            "source": f"{file_stem}.pdf",
            "format": "mineru+text-llm",
            "background": q["background"],
            "question": q["question"],
            "options": q["options"],
            "answer": q["answer"],
            "reference": q["reference"],
            "has_image": bool(q["image_paths"]),
            "image_paths": q["image_paths"],
        })
    tqdm.write(f"[{file_stem}] done — {len(questions)} question(s) extracted")
    return questions

print("Extractor defined.")

In [ ]:
import time

out_dir = Path(OUTPUT_FILE).parent
pdf_files = sorted(Path(PDFS_PATH).glob("*.pdf"))

if not pdf_files:
    print(f"No PDFs found in {INPUT_DIR}")
else:
    print(f"Found {len(pdf_files)} PDF files\n")

all_questions = []
run_start = time.monotonic()

for i, pdf in enumerate(tqdm(pdf_files, desc="PDFs", unit="file"), 1):
    tqdm.write(f"\n=== [{i}/{len(pdf_files)}] {pdf.name} ===")
    pdf_start = time.monotonic()
    qs    = extract_pdf(str(pdf), out_dir)
    pdf_dt = time.monotonic() - pdf_start
    auto  = sum(1 for q in qs if q["answer"])
    imgs  = sum(1 for q in qs if q["has_image"])
    fmt   = qs[0]["format"] if qs else "?"
    tqdm.write(f"  {pdf.name:<45}  Fmt:{fmt}  {len(qs):>2}Q  "
               f"{auto:>2} w/answer  {imgs} img  ({pdf_dt:.1f}s)")
    all_questions.extend(qs)

total_dt = time.monotonic() - run_start
print(f"\nAll PDFs processed in {total_dt:.1f}s")

# save
Path(OUTPUT_FILE).write_text(
    json.dumps(all_questions, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

total  = len(all_questions)
w_ans  = sum(1 for q in all_questions if q["answer"])
no_ans = total - w_ans
imgs   = sum(1 for q in all_questions if q["has_image"])

print(f"""
{'='*50}
Total questions  : {total}
With answer      : {w_ans}
Without answer   : {no_ans}  ← fill manually or use LLM
With images      : {imgs}
Output           : {OUTPUT_FILE}
{'='*50}
""")

In [ ]:
for q in all_questions[:3]:
    print(f"[{q['id']}]  fmt:{q['format']}  answer:{q['answer']}")
    if q["background"]:
        print(f"  BG: {q['background'][:100]}...")
    print(f"  Q : {q['question'][:100]}")
    for k, v in q["options"].items():
        marker = " ◀" if k == q["answer"] else ""
        print(f"    {k}. {v[:60]}{marker}")
    print()
